In [16]:
import os
import hashlib
from pathlib import Path
from PIL import Image, ImageOps

def validate_and_preprocess_image(image_path: str, output_dir: str, target_size: tuple = (512, 512)) -> dict:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {image_path}")

    file_size_kb = path.stat().st_size / 1024
    if file_size_kb > (10 * 1024):
        raise ValueError("File size exceeds the allowed limit.")

    sha256_hash = hashlib.sha256()
    with open(path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    checksum = f"sha256:{sha256_hash.hexdigest()}"

    os.makedirs(output_dir, exist_ok=True)
    derived_image_path = os.path.join(output_dir, f"{path.stem}_processed{path.suffix}")

    try:
        with Image.open(path) as img:
            width, height = img.size
            img_format = img.format.lower()

            if img_format not in ["jpeg", "png"]:
                raise ValueError(f"Unsupported image format: {img_format}")

            if width < 64 or height < 64:
                raise ValueError("Image dimensions are too small.")

            img = ImageOps.exif_transpose(img)
            if img.mode != 'RGB':
                img = img.convert('RGB')
                
            img_resized = img.resize(target_size, Image.Resampling.LANCZOS)
            img_resized.save(derived_image_path, quality=95)

    except Exception as e:
        raise ValueError(f"Image processing failed: {str(e)}")

    return {
        "status": "SUCCESS",
        "originalPath": str(path),
        "derivedPath": derived_image_path,
        "checksum": checksum,
        "metadata": {"width": width, "height": height, "mimeType": f"image/{img_format}"},
        "preprocessingConfig": {"colorSpace": "RGB", "resizingRule": f"lanczos_{target_size[0]}x{target_size[1]}"}
    }


def load_and_process_dataset(base_dir: str = "datasets/samples/cv", output_dir: str = "datasets/processed/cv"):
    samples_path = Path(base_dir)
    print(f"Checking directory: {samples_path.absolute()}")
    
    if not samples_path.exists():
        print(f" Directory not found: {base_dir}")
        return []

    processed_results = []
    files_found = list(samples_path.glob("**/*.*"))
    print(f" Total files found in directory: {len(files_found)}")

    for img_path in files_found:
        if img_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            print(f" Processing sample: {img_path}")
            try:
                result = validate_and_preprocess_image(str(img_path), output_dir)
                processed_results.append(result)
            except Exception as e:
                print(f" Error processing sample {img_path}: {e}")

    print(f"Done Processed {len(processed_results)} images successfully.")
    return processed_results

In [17]:
load_and_process_dataset(
    base_dir="../datasets/samples/cv/FF-PSI-107/SPICE_GMT045_2009_jpegs",
    output_dir="../datasets/processed/cv"
)

Checking directory: c:\FCDS\Nasa\cv\..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs
 Total files found in directory: 10
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030336.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030340.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030344.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030351.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030355.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030359.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030363.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030368.jpg
 Processing sample: ..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs\ISS018E030372.jpg
 Proce

[{'status': 'SUCCESS',
  'originalPath': '..\\datasets\\samples\\cv\\FF-PSI-107\\SPICE_GMT045_2009_jpegs\\ISS018E030336.jpg',
  'derivedPath': '../datasets/processed/cv\\ISS018E030336_processed.jpg',
  'checksum': 'sha256:2be316662a14beb73b53b84a3fd3450cde62142748aa60f271cfc4cead6b6f60',
  'metadata': {'width': 3008, 'height': 2000, 'mimeType': 'image/jpeg'},
  'preprocessingConfig': {'colorSpace': 'RGB',
   'resizingRule': 'lanczos_512x512'}},
 {'status': 'SUCCESS',
  'originalPath': '..\\datasets\\samples\\cv\\FF-PSI-107\\SPICE_GMT045_2009_jpegs\\ISS018E030340.jpg',
  'derivedPath': '../datasets/processed/cv\\ISS018E030340_processed.jpg',
  'checksum': 'sha256:b8489a93823ec4bbfa6979a5d0eb588222dd15d39f07f44b8d621c6f843a0e3e',
  'metadata': {'width': 3008, 'height': 2000, 'mimeType': 'image/jpeg'},
  'preprocessingConfig': {'colorSpace': 'RGB',
   'resizingRule': 'lanczos_512x512'}},
 {'status': 'SUCCESS',
  'originalPath': '..\\datasets\\samples\\cv\\FF-PSI-107\\SPICE_GMT045_2009_jpe

In [18]:
import os
import json
import hashlib
from pathlib import Path
from PIL import Image, ImageOps

def log_to_manifest(manifest_path: str, record: dict):
    """
    Appends the processing record to the JSONL dataset manifest file.
    """
    os.makedirs(os.path.dirname(manifest_path), exist_ok=True)
    with open(manifest_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

def validate_preprocess_and_manifest(image_path: str, output_dir: str, manifest_file: str, target_size: tuple = (512, 512)) -> dict:
    """
    Performs intake validation, preprocessing, baseline detector tagging, 
    and appends the result to the dataset manifest.
    """
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {image_path}")

    # 1. File size validation
    file_size_kb = path.stat().st_size / 1024
    if file_size_kb > (10 * 1024):  # Max limit 10MB
        raise ValueError("File size exceeds the allowed limit.")

    # 2. Compute SHA-256 checksum for data integrity
    sha256_hash = hashlib.sha256()
    with open(path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    checksum = f"sha256:{sha256_hash.hexdigest()}"

    # Create output directory for derived artifacts
    os.makedirs(output_dir, exist_ok=True)
    derived_image_path = os.path.join(output_dir, f"{path.stem}_processed{path.suffix}")

    # 3. Validation and Preprocessing
    try:
        with Image.open(path) as img:
            width, height = img.size
            img_format = img.format.lower()

            if img_format not in ["jpeg", "png"]:
                raise ValueError(f"Unsupported image format: {img_format}")

            if width < 64 or height < 64:
                raise ValueError("Image dimensions are too small.")

            # Correct orientation via EXIF and normalize color space
            img = ImageOps.exif_transpose(img)
            if img.mode != 'RGB':
                img = img.convert('RGB')
                
            # Resize with Lanczos resampling to preserve edge details
            img_resized = img.resize(target_size, Image.Resampling.LANCZOS)
            img_resized.save(derived_image_path, quality=95)

    except Exception as e:
        raise ValueError(f"Image processing failed: {str(e)}")

    # Construct manifest record including Baseline Detector selection
    record = {
        "imageId": path.stem,
        "originalPath": str(path),
        "derivedPath": derived_image_path,
        "checksumSha256": checksum,
        "status": "VALIDATED_AND_PREPROCESSED",
        "detectorBaseline": "Contour-Based-Thresholding-v1.0",
        "metadata": {
            "width": width,
            "height": height,
            "mimeType": f"image/{img_format}"
        }
    }

    # Log to manifest file
    log_to_manifest(manifest_file, record)
    return record


def run_baseline_pipeline(base_dir: str, output_dir: str, manifest_file: str):
    """
    Runs the pipeline over the target folder and records entries into the manifest.
    """
    samples_path = Path(base_dir)
    print(f" Checking directory: {samples_path.absolute()}")
    
    if not samples_path.exists():
        print(f" Directory not found: {base_dir}")
        return []

    processed_results = []
    files_found = list(samples_path.glob("**/*.*"))
    print(f" Total files found: {len(files_found)}")

    for img_path in files_found:
        if img_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            print(f" Processing: {img_path.name}")
            try:
                res = validate_preprocess_and_manifest(str(img_path), output_dir, manifest_file)
                processed_results.append(res)
            except Exception as e:
                print(f" Error on {img_path.name}: {e}")

    print(f" Completed! Successfully processed and logged {len(processed_results)} items.")
    return processed_results

In [19]:
run_baseline_pipeline(
    base_dir="../datasets/samples/cv/FF-PSI-107/SPICE_GMT045_2009_jpegs",
    output_dir="../datasets/processed/cv",
    manifest_file="../datasets/manifest.jsonl"
)

 Checking directory: c:\FCDS\Nasa\cv\..\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs
 Total files found: 10
 Processing: ISS018E030336.jpg
 Processing: ISS018E030340.jpg
 Processing: ISS018E030344.jpg
 Processing: ISS018E030351.jpg
 Processing: ISS018E030355.jpg
 Processing: ISS018E030359.jpg
 Processing: ISS018E030363.jpg
 Processing: ISS018E030368.jpg
 Processing: ISS018E030372.jpg
 Processing: ISS018E030376.jpg
 Completed! Successfully processed and logged 10 items.


[{'imageId': 'ISS018E030336',
  'originalPath': '..\\datasets\\samples\\cv\\FF-PSI-107\\SPICE_GMT045_2009_jpegs\\ISS018E030336.jpg',
  'derivedPath': '../datasets/processed/cv\\ISS018E030336_processed.jpg',
  'checksumSha256': 'sha256:2be316662a14beb73b53b84a3fd3450cde62142748aa60f271cfc4cead6b6f60',
  'status': 'VALIDATED_AND_PREPROCESSED',
  'detectorBaseline': 'Contour-Based-Thresholding-v1.0',
  'metadata': {'width': 3008, 'height': 2000, 'mimeType': 'image/jpeg'}},
 {'imageId': 'ISS018E030340',
  'originalPath': '..\\datasets\\samples\\cv\\FF-PSI-107\\SPICE_GMT045_2009_jpegs\\ISS018E030340.jpg',
  'derivedPath': '../datasets/processed/cv\\ISS018E030340_processed.jpg',
  'checksumSha256': 'sha256:b8489a93823ec4bbfa6979a5d0eb588222dd15d39f07f44b8d621c6f843a0e3e',
  'status': 'VALIDATED_AND_PREPROCESSED',
  'detectorBaseline': 'Contour-Based-Thresholding-v1.0',
  'metadata': {'width': 3008, 'height': 2000, 'mimeType': 'image/jpeg'}},
 {'imageId': 'ISS018E030344',
  'originalPath': '.

In [20]:
import cv2
import numpy as np
from pathlib import Path

def generate_segmentation_prototype(derived_image_path: str, mask_output_dir: str) -> dict:
    """
    Creates a segmentation prototype mask and extracts basic geometric features 
    (perimeter, circularity, aspectRatio) for the flame in microgravity.
    """
    path = Path(derived_image_path)
    if not path.exists():
        raise FileNotFoundError(f"Derived image not found: {derived_image_path}")

    # Read the preprocessed image using OpenCV
    img = cv2.imread(str(path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1. Apply thresholding to isolate the bright flame from the dark background
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)

    # 2. Find contours of the flame
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        # Fallback if no flame contour is clearly detected
        return {
            "maskStatus": "EMPTY_MASK",
            "geometry": {"perimeter": 0.0, "aspectRatio": 0.0, "circularity": 0.0}
        }

    # Select the largest contour corresponding to the main flame body
    largest_contour = max(contours, key=cv2.contourArea)

    # 3. Compute geometric properties
    perimeter = float(cv2.arcLength(largest_contour, True))
    area = cv2.contourArea(largest_contour)

    # Bounding box for Aspect Ratio
    x, y, w, h = cv2.boundingRect(largest_contour)
    aspect_ratio = float(w) / float(h) if h > 0 else 0.0

    # Circularity formula: (4 * pi * Area) / (Perimeter^2)
    circularity = float((4 * np.pi * area) / (perimeter ** 2)) if perimeter > 0 else 0.0

    # 4. Save the binary mask artifact
    os.makedirs(mask_output_dir, exist_ok=True)
    mask_filename = f"{path.stem}_mask.png"
    mask_path = os.path.join(mask_output_dir, mask_filename)
    
    # Generate binary mask image (white flame, black background)
    mask_img = np.zeros_like(gray)
    cv2.drawContours(mask_img, [largest_contour], -1, (255), thickness=cv2.FILLED)
    cv2.imwrite(mask_path, mask_img)

    return {
        "maskStatus": "SUCCESS",
        "maskType": "binary",
        "maskUri": mask_path,
        "geometry": {
            "perimeter": round(perimeter, 2),
            "aspectRatio": round(aspect_ratio, 2),
            "circularity": round(circularity, 2)
        }
    }

In [ ]:
# Segmentation 
sample_processed_image = "../datasets/processed/cv/ISS018E030336_processed.jpg"
seg_result = generate_segmentation_prototype(sample_processed_image, "../datasets/processed/masks")
print(seg_result)

{'maskStatus': 'SUCCESS', 'maskType': 'binary', 'maskUri': '../datasets/processed/masks\\ISS018E030336_processed_mask.png', 'geometry': {'perimeter': 406.75, 'aspectRatio': 4.07, 'circularity': 0.42}}


In [23]:
from pydantic import BaseModel
from typing import Optional, Dict

class AnalysisRequest(BaseModel):
    requestId: str
    imageId: str
    imagePath: str
    options: Optional[Dict[str, bool]] = {"runSegmentation": True, "extractGeometry": True}

class SegmentationResult(BaseModel):
    maskType: str = "binary"
    maskUri: Optional[str] = None
    maskChecksum: Optional[str] = None
    overlayArtifactUri: Optional[str] = None

class GeometryResult(BaseModel):
    perimeter: float
    aspectRatio: float
    circularity: float

class AnalysisResponse(BaseModel):
    requestId: str
    status: str
    segmentation: SegmentationResult
    geometry: GeometryResult